In [77]:
!pip install -q git+https://github.com/PyThaiNLP/pythainlp

^C


In [ ]:
# install pythainlp and ssg(subword tokenizer)
!pip install -q ssg

In [99]:
from typing import List, Union
from pythainlp.tokenize import subword_tokenize,word_tokenize
from pythainlp.util import sound_syllable
from pythainlp.util import remove_tonemark
from pythainlp.khavee import KhaveeVerifier
import pythainlp as pythai
from pythainlp.tokenize import word_tokenize
from pythainlp.tokenize import subword_tokenize
from pythainlp.util import sound_syllable
from pythainlp.util import isthai
from pythainlp.transliterate import pronunciate
from pythainlp.spell import correct
from tqdm import tqdm
import numpy as np
import pandas as pd
kv = KhaveeVerifier()

### Word and Subword Tokenizing

In [ ]:
# split text from \n to list and drop soi word ->  splitted wak list (no soi)
def split_klong(klong_text):
  splitted_klong = []
  klong_list = klong_text.split('\n')
  klong_list = [klong for klong in klong_list if klong.strip()]
  for i in range(len(klong_list)):
    if i == 1 or i == 3 or i == 5:
      klong = klong_list[i]
      if klong[0] == ' ':
        klong = klong[1:]
      klong = klong.split(' ')
      print(f"klong: {klong}")
      splitted_klong.append(klong[0])
    else:
      splitted_klong.append(klong_list[i].replace(' ', ''))
  return splitted_klong

In [ ]:
# subword tokenize wak with ssg and dict
def subword_token(wak, engine='ssg'):
  subword_tokenized = subword_tokenize(wak, engine='ssg')
  if len(subword_tokenized) != 5 and len(subword_tokenized) != 2:
      subword_tokenized = subword_tokenize(wak, engine='dict')
  return subword_tokenized

In [ ]:
klong_txt = """เสียงลือเสียงเล่าอ้าง
อันใด พี่เอย
เสียงย่อมยอยศใคร
ทั่วหล้า
สองเขือพี่หลับไหล
ลืมตื่น ฤๅพี่
สองพี่คิดเองอ้า
อย่าได้ถามเผือ"""
klong_txt2 = """พระสมุทรสุดลึกล้น
คณนา
สายดิ่งทิ้งทอดมา
หยั่งได้
เขาสูงอาจวัดวา
กำหนด
จิตมนุษย์นี้ไซร้
ยากแท้หยั่งถึง
"""
print(klong_txt2+"\n\n")
splitted_klong = split_klong(klong_txt2)
print(f"splitted_klong: {splitted_klong}\n\n")
for i in range(len(splitted_klong)):
  wak = splitted_klong[i]
  subword_tokenized = subword_token(wak)
  print(f"wak: {wak} -> subword tokenized: {subword_tokenized}")

พระสมุทรสุดลึกล้น
คณนา
สายดิ่งทิ้งทอดมา
หยั่งได้
เขาสูงอาจวัดวา
กำหนด
จิตมนุษย์นี้ไซร้
ยากแท้หยั่งถึง



klong: ['คณนา']
klong: ['หยั่งได้']
klong: ['กำหนด']
splitted_klong: ['พระสมุทรสุดลึกล้น', 'คณนา', 'สายดิ่งทิ้งทอดมา', 'หยั่งได้', 'เขาสูงอาจวัดวา', 'กำหนด', 'จิตมนุษย์นี้ไซร้', 'ยากแท้หยั่งถึง']


wak: พระสมุทรสุดลึกล้น -> subword tokenized: ['พระ', 'สมุทร', 'สุด', 'ลึก', 'ล้น']
wak: คณนา -> subword tokenized: ['คณ', 'นา']
wak: สายดิ่งทิ้งทอดมา -> subword tokenized: ['สาย', 'ดิ่ง', 'ทิ้ง', 'ทอด', 'มา']
wak: หยั่งได้ -> subword tokenized: ['หยั่ง', 'ได้']
wak: เขาสูงอาจวัดวา -> subword tokenized: ['เขา', 'สูง', 'อาจ', 'วัด', 'วา']
wak: กำหนด -> subword tokenized: ['กำ', 'หนด']
wak: จิตมนุษย์นี้ไซร้ -> subword tokenized: ['จิต', 'มนุษย์', 'นี้', 'ไซ', 'ร้']
wak: ยากแท้หยั่งถึง -> subword tokenized: ['ยาก', 'แท้', 'หยั่ง', 'ถึง']


### Check Functions

#### Number of syllables check

In [ ]:

# check number of syllables -> [True, True, True, True, True, True, True, True] (len=8)
def subword_num(splitted_klong):
  checked = []
  two = [1,3,5]
  five = [0,2,4,6]
  for num in range(len(splitted_klong)):
    if num in two:
      checked.append(len(subword_token(splitted_klong[num])) == 2)
    elif num in five:
      checked.append(len(subword_token(splitted_klong[num])) == 5)
    elif num == 7:
      checked.append(len(subword_token(splitted_klong[num])) == 4)
  return checked

#### eak tou check


In [ ]:
# check what word tone is
def find_tone(word):
  char_list = [*word]
  if "่" in char_list or sound_syllable(word) == 'dead':
    return "eak or dead"
  elif "้" in char_list:
    return "tou"
  else:
    return False

In [ ]:
# check eaktou -> list[True, True, True, True, True, True, True, True] (len=8)
def check_eaktou(splitted_klong):
  checked = []
  for num in range(len(splitted_klong)):
    tokenzied_wak = subword_token(splitted_klong[num])
    if num == 0:
      checked.append(find_tone(tokenzied_wak[3]) == "eak or dead" and find_tone(tokenzied_wak[4]) == 'tou')
    elif num == 1:
      checked.append(True)
    elif num == 2:
      checked.append(find_tone(tokenzied_wak[1]) == "eak or dead")
    elif num == 3:
      checked.append(find_tone(tokenzied_wak[0]) == 'eak or dead' and find_tone(tokenzied_wak[1]) == 'tou')
    elif num == 4:
      checked.append(find_tone(tokenzied_wak[2]) == 'eak or dead')
    elif num == 5:
      checked.append(find_tone(tokenzied_wak[1]) == 'eak or dead')
    elif num == 6:
      checked.append(find_tone(tokenzied_wak[1]) == "eak or dead" and find_tone(tokenzied_wak[4]) == 'tou')
    elif num == 7:
      checked.append(find_tone(tokenzied_wak[0]) == "eak or dead" and find_tone(tokenzied_wak[1]) == 'tou')
  return checked

#### sampas check

In [100]:
# last sound of wak from pronunciate tokenized last word of each wak
# ex [เสียงลือเสียงเล่าอ้าง] -> [อ้าง]
def sound_words(splitted_klong):
  sound_list = []
  for wak in splitted_klong:
    list_char = [*wak]
    if " " in list_char:
      wak = wak.split(" ")
      wak = wak[0]
    wak = word_tokenize(wak, engine="newmm")
    pronounce_word = pronunciate(wak[-1], engine="w2p")
    sound_list.append(pronounce_word.replace('ฺ', '').split('-')[-1])
  return sound_list

In [101]:
# check sampas -> [True, True, True]
# [0] = sampas wak 2-3, [1] = sampas wak 2-4, [2] sampas wak 4-7
def check_sampas(sound_list):
  checked = []
  if len(sound_list) > 2:
    checked.append(kv.check_sumpus(sound_list[1],sound_list[2]))
    if len(sound_list) > 4:
      checked.append(kv.check_sumpus(sound_list[1],sound_list[4]))
      if len(sound_list) > 6:
        checked.append(kv.check_sumpus(sound_list[3],sound_list[6]))
  else:
    checked.append(True)
  return checked

#### Main Check

In [ ]:
def main_check(klong_text):
  splitted_klong = split_klong(klong_text)
  checked_subword_num = subword_num(splitted_klong)
  if False in checked_subword_num:
    false_index = checked_subword_num.index(False)
    return 'syllable format error', false_index+1
  else:
    checked_eaktou = check_eaktou(splitted_klong)
    if False in checked_eaktou:
      false_index = checked_eaktou.index(False)
      return 'eaktou format error', false_index+1
    else:
      sound_list = sound_words(splitted_klong)
      checked_sampas = check_sampas(sound_list)
      if False in checked_sampas:
        wak_sampas = ['2 and 3', '2 and 5', '4 and 7']
        return 'sampas format error', wak_sampas[checked_sampas.index(False)]
      else:
        return True

In [ ]:
def analyze_wak(wak_text):
    # 1. Tokenize into words
    words = word_tokenize(wak_text, engine="newmm")
    
    wak_data = []
    total_syllables = 0
    
    # 2. Analyze each word
    for word in words:
        # Ignore whitespace
        if word.strip() == "":
            continue
            
        spoken = pronunciate(word, engine="w2p")
        syllables = spoken.split("-")
        count = len(syllables)
        
        wak_data.append({
            "original_word": word,
            "spoken_form": spoken,
            "syllable_count": count
        })
        
        total_syllables += count

    # 3. Classify the Wak based on your rules
    classification = ""
    target_rhythm = []
    needs_manual_review = False
    
    if total_syllables == 9:
        classification = "9 Syllables"
        target_rhythm = [3, 3, 3]
    elif total_syllables == 8:
        classification = "8 Syllables"
        target_rhythm = [3, 2, 3]
    elif total_syllables == 7:
        classification = "7 Syllables"
        # Flagging for manual review as requested
        target_rhythm = [3, 2, 2] # Default assumption
        needs_manual_review = True
    else:
        classification = f"Irregular ({total_syllables} Syllables)"
        needs_manual_review = True

    return {
        "total_syllables": total_syllables,
        "classification": classification,
        "rhythm": target_rhythm,
        "needs_review": needs_manual_review,
        "word_breakdown": wak_data
    }

# --- Example Usage ---
test_wak = "ธรรมชาติสวยงามตามภูผา" # ธรรมชาติ (3) สวยงาม (2) ตาม (1) ภูผา (2) = 8
test_wak2 = "ปรารถนากนิษฐภคินี"
test_wak3 = "อุปการคุณกเฬวรากสมรรถภาพกิตติมศักดิ์ปรากฏการณ์มัธยัสถ์ภูมิลำเนามนุษย์อินทคาม ก ธ ร ฤ ลืมตื่นฤๅพี่"
test_wak4 = "อัธยาศัยโปร่งใสไสยศาสตร์วิสัยสดใสสงสัย"
result = analyze_wak(test_wak)
result2 = analyze_wak(test_wak2)
result3 = analyze_wak(test_wak3)
result4 = analyze_wak(test_wak4)

print(f"Total: {result['total_syllables']} ({result['classification']})")
print(f"Rhythm: {result['rhythm']}")
print(f"Review Needed: {result['needs_review']}")
for item in result['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")
    
print(f"Total: {result2['total_syllables']} ({result2['classification']})")
print(f"Rhythm: {result2['rhythm']}")
print(f"Review Needed: {result2['needs_review']}")
for item in result2['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")

print(f"Total: {result3['total_syllables']} ({result3['classification']})")
print(f"Rhythm: {result3['rhythm']}")
print(f"Review Needed: {result3['needs_review']}")
for item in result3['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")
    
print(f"Total: {result4['total_syllables']} ({result4['classification']})")
print(f"Rhythm: {result4['rhythm']}")
print(f"Review Needed: {result4['needs_review']}")
for item in result4['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")

Total: 8 (8 Syllables)
Rhythm: [3, 2, 3]
Review Needed: False
 - ธรรมชาติ -> ทำ-มะ-ชาด (3 beats)
 - สวยงาม -> สวย-งาม (2 beats)
 - ตาม -> ตาม (1 beats)
 - ภูผา -> พู-ผา (2 beats)
Total: 9 (9 Syllables)
Rhythm: [3, 3, 3]
Review Needed: False
 - ปรารถนา -> ปราด-ถะ-หนา (3 beats)
 - กนิษฐภคินี -> กะ-นิด-ถะ-พะ-คะ-นี (6 beats)
Total: 41 (Irregular (41 Syllables))
Rhythm: []
Review Needed: True
 - อุปการคุณ -> อุบ-ปะ-กาน-คุน (4 beats)
 - กเฬวราก -> กะ-เล-วะ-ราก (4 beats)
 - สมรรถภาพ -> สะ-มะ-ถะ-พาบ (4 beats)
 - กิตติมศักดิ์ -> กิด-ติม-สัก (3 beats)
 - ปรากฏการณ์ -> ปฺรา-กด-กาน (3 beats)
 - มัธยัสถ์ -> มัด-ทะ-ยัด (3 beats)
 - ภูมิลำเนา -> พูม-ลำ-เนา (3 beats)
 - มนุษย์ -> มะ-นุด (2 beats)
 - อิน -> อิน (1 beats)
 - ท -> ทอด (1 beats)
 - คาม -> คาม (1 beats)
 - ก -> กะ-โหฺมด (2 beats)
 - ธ -> ทอน (1 beats)
 - ร -> ระ-คะ-แนน (3 beats)
 - ฤ -> รึ-เริฟ (2 beats)
 - ลืม -> ลืม (1 beats)
 - ตื่น -> ตื่น (1 beats)
 - ฤๅ -> รือ (1 beats)
 - พี่ -> พี่ (1 beats)
Total: 14 (Irregular (14 Syllables))
Rhy

In [ ]:
problematic_word = ["ฤๅ","ก" ,"ข", "ค", "ฆ", "ง", "จ", "ฉ", "ช", "ซ", "ฌ", "ญ", "ฎ", "ฏ", "ฐ", "ฑ", "ฒ", "ณ", "ด", "ต", "ถ", "ท", "ธ", "น", "บ", "ป", "ผ", "ฝ", "พ", "ฟ", "ภ", "ม", "ย", "ร", "ล", "ว", "ฤ"]
for word in problematic_word:
    result = pronunciate(word, engine="w2p")
    print(f"{word} -> {result}")

ฤๅ -> รือ
ก -> กะ-โหฺมด
ข -> ขอ
ค -> คอ-คอน
ฆ -> คะ-คะ-โท
ง -> ง
จ -> จอ-โจ
ฉ -> ฉะ-หะ-พัน
ช -> ชด-ชะ-สู
ซ -> ซะ-ซิด
ฌ -> ชด
ญ -> ยด
ฎ -> ด๊อย
ฏ -> ตะ-เดด
ฐ -> ถะ
ฑ -> ทด
ฒ -> ทัด-ถะ-สะ-ทะ
ณ -> นะ
ด -> ดด
ต -> ตำ-ตะ-โจด
ถ -> ถะ-ถอน
ท -> ทอด
ธ -> ทอน
น -> นะ-โจน
บ -> บอบ
ป -> ปอ-คอน
ผ -> ผะ-หฺมด
ฝ -> ฝี-ฝน
พ -> พบ
ฟ -> ฟ
ภ -> พะ-โถบ
ม -> มะ-โหฺม
ย -> ยด
ร -> ระ-คะ-แนน
ล -> ละ-โน
ว -> วะ-วิว
ฤ -> รึ-เริฟ


In [102]:
print(kv.check_klon(
    "แม่วันทองของลูกจงกลับบ้าน ขาจะพาลว้าวุ่นแม่ทูนหัว จะก้มหน้าลาไปมิได้กลัว แม่อย่ามัวหมองนักจงหักใจ",
    k_type=8
))

["Can't find rhyme between paragraphs ('บ้าน', ['จะ', 'พาล', 'ว้า', 'วุ่น']) in paragraph 1"]


In [106]:
print(kv.is_sumpus("บ้าน", "พาล"))
print(kv.is_sumpus("กัย", "กัย"))

False
False


In [105]:
print(kv.check_sara("พาล"))
print(kv.check_marttra("พาล"))
print(kv.check_marttra("พาน"))

อา
เกย
กน


In [91]:
print(kv._has_true_final_yl("ล"))

False


In [1]:
from pythainlp.tokenize import word_tokenize, syllable_tokenize
from pythainlp.transliterate import pronunciate

# 1. THE GOLD STANDARD DICTIONARY
POETRY_OVERRIDES = {
    "ก็": ["ก็"],
    "บ่": ["บ่"],
    "ได้": ["ได้"],
    "ธ": ["ทะ"],
    "ณ": ["นะ"],
    "ฤ": ["รึ"],
    "ฤๅ": ["รือ"],
    "ฤๅษี": ["รือ", "สี"],
    "มนุษย์": ["มะ", "นุด"],
    "พฤติกรรม": ["พฺรึด", "ติ", "กำ"],
    "สระ": ["สะ", "หระ"],
    "ความรู้สึก": ["ความ", "รู้", "สึก"],
    "เข้าใจ": ["เข้า", "ใจ"],
    "อยู่": ["อยู่"], # Fixes the "รู้อยู่" -> 'หยู่' hallucination
    "ผู้": ["ผู้"],   # Fixes the "ผู้ใด" -> 'พู่' hallucination
    "ใหม่": ["ใหม่"],
    "อย่า": ["อย่า"],
    "อยู่": ["อยู่"],
    "อย่าง": ["อย่าง"],
    "อยาก": ["อยาก"],
}

def process_w2p(word: str) -> list:
    """Helper function to cleanly process a string through the w2p model."""
    # pronunciate hallucination on very short words. Use syllable_tokenize instead.
    if len(word) <= 2:
        return syllable_tokenize(word, engine="ssg")
        
    phonetic_word = pronunciate(word, engine="w2p")
    
    if not phonetic_word:
        return syllable_tokenize(word, engine="ssg")
    
    # Clean up unwanted characters like Phinthu (-ฺ) and hyphens (-)
    clean_phonetic = phonetic_word.replace("ฺ", "").replace("-", "")
    
    # Using ssg (CRF segmenter) as it handles phonetic text better than dict
    phonetic_syllables = syllable_tokenize(clean_phonetic, engine="ssg")
    
    # Clean up stray 'ห' artifacts
    return [s for s in phonetic_syllables if s != "ห" or word == "ห"]

def extract_poetic_syllables(text: str) -> list:
    """Extracts phonetic syllables for Klon 8 verification."""
    # Tokenize text into words using the newmm (greedy) engine, allowing w2p to handle compound words
    words = word_tokenize(text, engine="newmm")
    final_syllables = []
    
    for word in words:
        # Direct Override if the tokenized word is in the POETRY_OVERRIDES list (O(1) Fast Lookup)
        if word in POETRY_OVERRIDES:
            final_syllables.extend(POETRY_OVERRIDES[word])
            continue

        # Use 'ssg' to check if newmm greedy engine merged words like "ได้ใจ" or "รู้อยู่"
        sub_syllables = syllable_tokenize(word, engine="ssg")
        
        # Check if ANY of the segmented syllables are in the POETRY_OVERRIDES list
        has_override = any(sub in POETRY_OVERRIDES for sub in sub_syllables)
        
        if len(sub_syllables) > 1 and has_override:
            # Found a hidden override word in the segmented syllables. Process each sub-syllable independently.
            for sub in sub_syllables:
                if sub in POETRY_OVERRIDES:
                    final_syllables.extend(POETRY_OVERRIDES[sub])
                else:
                    final_syllables.extend(process_w2p(sub))
            continue
        
        # If no overrides were found inside, treat it as a true compound word (e.g. พัฒนาการ)
        # Proceed to parse the word into w2p pronunciate engine.
        final_syllables.extend(process_w2p(word))
        
    return final_syllables

# Test sentences
sentence = [
    "แม่รักลูกลูกก็รู้อยู่ว่ารัก",
    "สรรเพชญโพธิญาณประมาณหมาย", 
    "บ่มีผู้ใดจะเข้าใจความรู้สึกของข้าได้",
    "ไหนใครใคร่ใช้ได้ใจชัยไทย",
    "โอ้มนุษย์ผู้มีจิตใจและมีความสามารถในการสร้างสรรค์สิ่งใหม่",
    "ความแปลกแยกและพัฒนาการ",
    "อย่าอยู่อย่างอยากหมากหมิ่นหมายหมอง",
    "อันประกอบด้วยสระและพยัญชนะหลายตัว",
]

for sent in sentence:
    syllables = extract_poetic_syllables(sent)
    print(f"Sentence: {sent}")
    print(f"Final Syllable: {syllables}\n")

Sentence: แม่รักลูกลูกก็รู้อยู่ว่ารัก
Final Syllable: ['แม่', 'รัก', 'ลูก', 'ลูก', 'ก็', 'รู้', 'อยู่', 'ว่า', 'รัก']

Sentence: สรรเพชญโพธิญาณประมาณหมาย
Final Syllable: ['สัน', 'เพ็ด', 'โพ', 'ทิ', 'ยาน', 'ประ', 'มาน', 'หมาย']

Sentence: บ่มีผู้ใดจะเข้าใจความรู้สึกของข้าได้
Final Syllable: ['บ่', 'มี', 'ผู้', 'ใด', 'จะ', 'เข้า', 'ใจ', 'ความ', 'รู้', 'สึก', 'ของ', 'ข้า', 'ได้']

Sentence: ไหนใครใคร่ใช้ได้ใจชัยไทย
Final Syllable: ['ใหน', 'ไคร', 'ไค', 'ร่', 'ไช้', 'ได้', 'ใจ', 'ไช', 'ไท']

Sentence: โอ้มนุษย์ผู้มีจิตใจและมีความสามารถในการสร้างสรรค์สิ่งใหม่
Final Syllable: ['โอ้', 'มะ', 'นุด', 'ผู้', 'มี', 'จิด', 'ใจ', 'และ', 'มี', 'ความ', 'สา', 'มาด', 'ใน', 'กาน', 'ส้าง', 'สัน', 'สิ่ง', 'ใหม่']

Sentence: ความแปลกแยกและพัฒนาการ
Final Syllable: ['ความ', 'แปลก', 'แยก', 'และ', 'พัด', 'ทะ', 'นา', 'กาน']

Sentence: อย่าอยู่อย่างอยากหมากหมิ่นหมายหมอง
Final Syllable: ['อย่า', 'อยู่', 'อย่าง', 'อยาก', 'หมาก', 'หมิ่น', 'หมาย', 'หมอง']

Sentence: อันประกอบด้วยสระและพยัญชนะหลายตัว
Final Syllable: ['